In [ ]:
%pip install s3fs web3

In [3]:
# =============== fetch_events_web3.py
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import s3fs
from google.colab import userdata
from web3 import Web3

# =============== fetch_transfers_etherscan.py & fetch_eth_etherscan.py

import random
import threading
import pyarrow as pa
import pyarrow.parquet as pq
import requests

# =============== preprocess_events_and_transfers.py

import pyarrow.compute as pc

# =============== trace_wallets.py

from collections import defaultdict



fetch_events_web3.py

In [ ]:
S3_ENDPOINT = "https://minio.lab.sspcloud.fr"

def load_credentials():
    """Read the Alchemy key and the Onyxia S3 keys from Colab secrets"""
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID").strip()
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY").strip()
    os.environ["AWS_SESSION_TOKEN"]     = userdata.get("AWS_SESSION_TOKEN").strip()
    os.environ["AWS_ENDPOINT_URL"]      = S3_ENDPOINT
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
    return userdata.get("ALCHEMY_API_KEY").strip()

POOL_ADDRESS = "0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2"  # Aave V3 Pool (proxy)
# START_BLOCK  = 0xF89537   # 16,291,127: Pool proxy deployment
START_BLOCK = 0x1808580 # 25,200,000: used only for testing
END_BLOCK = 0x180CEED  # 25,218,797

S3_ROOT      = "s3://bastienlevyguinot/aave_v3"
OUT_DIR      = f"{S3_ROOT}/aave_v3_events_parts"
OUT_FILE     = "s3://bastienlevyguinot/diffusion/aave_v3_events.parquet"
RESERVE_DIR  = f"{S3_ROOT}/aave_v3_reserve_parts"
RESERVE_FILE = f"{S3_ROOT}/aave_v3_reserve_data.parquet"

CHUNK_SIZE   = 10      # blocks per eth_getLogs call (Alchemy free plan max: 10)
SEGMENT_SIZE = 20_000  # blocks per saved Parquet part
WORKERS      = 4       # parallel eth_getLogs calls

POOL_EVENTS_ABI = [
    {
        "anonymous": False, "type": "event", "name": "Supply",
        "inputs": [
            {"indexed": True,  "name": "reserve",      "type": "address"},
            {"indexed": False, "name": "user",         "type": "address"},
            {"indexed": True,  "name": "onBehalfOf",   "type": "address"},
            {"indexed": False, "name": "amount",       "type": "uint256"},
            {"indexed": True,  "name": "referralCode", "type": "uint16"},
        ],
    },
    {
        "anonymous": False, "type": "event", "name": "Withdraw",
        "inputs": [
            {"indexed": True,  "name": "reserve", "type": "address"},
            {"indexed": True,  "name": "user",    "type": "address"},
            {"indexed": True,  "name": "to",      "type": "address"},
            {"indexed": False, "name": "amount",  "type": "uint256"},
        ],
    },
    {
        "anonymous": False, "type": "event", "name": "Borrow",
        "inputs": [
            {"indexed": True,  "name": "reserve",          "type": "address"},
            {"indexed": False, "name": "user",             "type": "address"},
            {"indexed": True,  "name": "onBehalfOf",       "type": "address"},
            {"indexed": False, "name": "amount",           "type": "uint256"},
            {"indexed": False, "name": "interestRateMode", "type": "uint8"},
            {"indexed": False, "name": "borrowRate",       "type": "uint256"},
            {"indexed": True,  "name": "referralCode",     "type": "uint16"},
        ],
    },
    {
        "anonymous": False, "type": "event", "name": "Repay",
        "inputs": [
            {"indexed": True,  "name": "reserve",    "type": "address"},
            {"indexed": True,  "name": "user",       "type": "address"},
            {"indexed": True,  "name": "repayer",    "type": "address"},
            {"indexed": False, "name": "amount",     "type": "uint256"},
            {"indexed": False, "name": "useATokens", "type": "bool"},
        ],
    },
    {
        "anonymous": False, "type": "event", "name": "LiquidationCall",
        "inputs": [
            {"indexed": True,  "name": "collateralAsset",            "type": "address"},
            {"indexed": True,  "name": "debtAsset",                  "type": "address"},
            {"indexed": True,  "name": "user",                       "type": "address"},
            {"indexed": False, "name": "debtToCover",                "type": "uint256"},
            {"indexed": False, "name": "liquidatedCollateralAmount", "type": "uint256"},
            {"indexed": False, "name": "liquidator",                 "type": "address"},
            {"indexed": False, "name": "receiveAToken",              "type": "bool"},
        ],
    },
    # Fetching ReserveDataUpdated events will be useful for later analysis
    # It won't be used in this project
    {
        "anonymous": False, "type": "event", "name": "ReserveDataUpdated",
        "inputs": [
            {"indexed": True,  "name": "reserve",             "type": "address"},
            {"indexed": False, "name": "liquidityRate",       "type": "uint256"},
            {"indexed": False, "name": "stableBorrowRate",    "type": "uint256"},
            {"indexed": False, "name": "variableBorrowRate",  "type": "uint256"},
            {"indexed": False, "name": "liquidityIndex",      "type": "uint256"},
            {"indexed": False, "name": "variableBorrowIndex", "type": "uint256"},
        ],
    },
]

# Common columns are filled for every event; the others only for their event type.
# "user" is the account whose position changes (onBehalfOf for Supply and Borrow).
# uint256 values are kept exact as strings in *_raw columns (convert with int())
COLUMN_DTYPES = {
    "event_name":                   "string",
    "block_number":                 "Int64",
    "tx_hash":                      "string",
    "log_index":                    "Int64",
    "reserve":                      "string",
    "user":                         "string",
    "amount_raw":                   "string",
    # Supply / Borrow
    "caller":                       "string",
    "referral_code":                "Int64",
    # Withdraw
    "recipient":                    "string",
    # Borrow
    "interest_rate_mode":           "Int64",
    "borrow_rate_raw":              "string",
    # Repay
    "repayer":                      "string",
    "use_atokens":                  "boolean",
    # LiquidationCall
    "collateral_asset":             "string",
    "liquidated_collateral_raw":    "string",
    "liquidator":                   "string",
    "receive_atoken":               "boolean",
}

# ReserveDataUpdated: the reserve's state right after an action. Rates are yearly,
# indexes are cumulative interest factors; all are in ray units (1e27 = 1.0)
RESERVE_COLUMN_DTYPES = {
    "block_number":                 "Int64",
    "tx_hash":                      "string",
    "log_index":                    "Int64",
    "reserve":                      "string",
    "liquidity_rate_raw":           "string",
    "stable_borrow_rate_raw":       "string",
    "variable_borrow_rate_raw":     "string",
    "liquidity_index_raw":          "string",
    "variable_borrow_index_raw":    "string",
}

PART_RE = re.compile(r"part_(\d+)_(\d+)\.parquet$")


def get_logs(w3, topics, from_block, to_block, max_retries=8):
    """eth_getLogs with backoff on rate limits, splitting the range if the provider refuses it."""
    for attempt in range(max_retries):
        try:
            return w3.eth.get_logs({
                "address":   POOL_ADDRESS,
                "fromBlock": from_block,
                "toBlock":   to_block,
                "topics":    [topics],
            })
        except Exception as e:
            msg = str(e).lower()
            if ("block range" in msg or "response size" in msg) and to_block > from_block:
                mid = (from_block + to_block) // 2
                return (get_logs(w3, topics, from_block, mid) + get_logs(w3, topics, mid + 1, to_block))
            if attempt == max_retries - 1:
                raise RuntimeError(f"eth_getLogs failed for blocks {from_block}-{to_block}: {e}") from e
            time.sleep(min(2 ** attempt, 60))


def decode(log, events_by_topic):
    """Decodes log into human-readable dict, format changes in function of the event type."""
    event = events_by_topic[Web3.to_hex(log["topics"][0])]
    ev = event.process_log(log)
    a = ev["args"]
    row = {
        "event_name":   ev["event"],
        "block_number": ev["blockNumber"],
        "tx_hash":      Web3.to_hex(ev["transactionHash"]),
        "log_index":    ev["logIndex"],
    }
    if ev["event"] == "Supply":
        row.update(
            reserve=a["reserve"], user=a["onBehalfOf"], amount_raw=str(a["amount"]),
            caller=a["user"], referral_code=a["referralCode"],
        )
    elif ev["event"] == "Withdraw":
        row.update(
            reserve=a["reserve"], user=a["user"], amount_raw=str(a["amount"]),
            recipient=a["to"],
        )
    elif ev["event"] == "Borrow":
        row.update(
            reserve=a["reserve"], user=a["onBehalfOf"], amount_raw=str(a["amount"]),
            caller=a["user"], interest_rate_mode=a["interestRateMode"],
            borrow_rate_raw=str(a["borrowRate"]), referral_code=a["referralCode"],
        )
    elif ev["event"] == "Repay":
        row.update(
            reserve=a["reserve"], user=a["user"], amount_raw=str(a["amount"]),
            repayer=a["repayer"], use_atokens=a["useATokens"],
        )
    elif ev["event"] == "LiquidationCall":
        row.update(
            reserve=a["debtAsset"], user=a["user"], amount_raw=str(a["debtToCover"]),
            collateral_asset=a["collateralAsset"],
            liquidated_collateral_raw=str(a["liquidatedCollateralAmount"]),
            liquidator=a["liquidator"], receive_atoken=a["receiveAToken"],
        )
    else:  # ReserveDataUpdated
        row.update(
            reserve=a["reserve"],
            liquidity_rate_raw=str(a["liquidityRate"]),
            stable_borrow_rate_raw=str(a["stableBorrowRate"]),
            variable_borrow_rate_raw=str(a["variableBorrowRate"]),
            liquidity_index_raw=str(a["liquidityIndex"]),
            variable_borrow_index_raw=str(a["variableBorrowIndex"]),
        )
    return row


def to_frame(rows, dtypes):
    return pd.DataFrame(rows, columns=list(dtypes)).astype(dtypes)

def list_parts(fs, out_dir):
    """s3:// paths of the saved segments in out_dir."""
    fs.invalidate_cache(out_dir)
    return sorted(f"s3://{p}" for p in fs.glob(f"{out_dir}/part_*.parquet"))

def resume_block(fs, out_dir, start):
    """First block not yet covered by a saved segment."""
    ends = [int(m.group(2)) for p in list_parts(fs, out_dir) if (m := PART_RE.search(p))]
    return max([start] + [e + 1 for e in ends])


def merge_parts(fs, out_dir, start, end, out_file):
    """from parts files to parquet in diffusion"""
    parts = [
        p for p in list_parts(fs, out_dir)
        if (m := PART_RE.search(p)) and int(m.group(1)) >= start and int(m.group(2)) <= end
    ]
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    df = df.sort_values(["block_number", "log_index"], ignore_index=True)
    df.to_parquet(out_file, index=False)
    return df


def main():
    api_key = load_credentials()
    w3 = Web3(Web3.HTTPProvider(f"https://eth-mainnet.g.alchemy.com/v2/{api_key}",
                                request_kwargs={"timeout": 30}))
    if not w3.is_connected():
        raise SystemExit("Could not connect to Alchemy")

    end = min(END_BLOCK, w3.eth.block_number)
    pool = w3.eth.contract(address=POOL_ADDRESS, abi=POOL_EVENTS_ABI)
    events = [getattr(pool.events, abi["name"])() for abi in POOL_EVENTS_ABI]
    events_by_topic = {e.topic: e for e in events}
    topics = list(events_by_topic)

    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})  # keys from the AWS_* env vars set above
    first = min(resume_block(fs, OUT_DIR, START_BLOCK), resume_block(fs, RESERVE_DIR, START_BLOCK))
    if first > START_BLOCK:
        print(f"Resuming from block {first:,}")

    total_blocks = end - first + 1
    n_events = n_updates = 0
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for seg_start in range(first, end + 1, SEGMENT_SIZE):
            seg_end = min(seg_start + SEGMENT_SIZE - 1, end)
            chunks = [(a, min(a + CHUNK_SIZE - 1, seg_end))
                      for a in range(seg_start, seg_end + 1, CHUNK_SIZE)]

            logs = [log for batch in executor.map(lambda c: get_logs(w3, topics, *c), chunks) for log in batch]
            rows = [decode(log, events_by_topic) for log in logs]
            events_df  = to_frame([r for r in rows if r["event_name"] != "ReserveDataUpdated"], COLUMN_DTYPES)
            reserve_df = to_frame([r for r in rows if r["event_name"] == "ReserveDataUpdated"], RESERVE_COLUMN_DTYPES)
            part_name = f"part_{seg_start:09d}_{seg_end:09d}.parquet"
            events_df.to_parquet(f"{OUT_DIR}/{part_name}", index=False)
            reserve_df.to_parquet(f"{RESERVE_DIR}/{part_name}", index=False)

            n_events  += len(events_df)
            n_updates += len(reserve_df)
            done = seg_end - first + 1
            elapsed = time.time() - t0
            eta_h = elapsed / done * (total_blocks - done) / 3600
            print(f"[{done / total_blocks:6.1%}] blocks {seg_start:,}-{seg_end:,} | "
                  f"{len(events_df):5d} events (total {n_events:,}) | "
                  f"{len(reserve_df):5d} reserve updates (total {n_updates:,}) | ETA {eta_h:.1f} h", flush=True)

    events_df = merge_parts(fs, OUT_DIR, START_BLOCK, end, OUT_FILE)
    print(f"Saved {len(events_df):,} events to {OUT_FILE}")
    print(events_df["event_name"].value_counts().to_string())

    reserve_df = merge_parts(fs, RESERVE_DIR, START_BLOCK, end, RESERVE_FILE)
    print(f"Saved {len(reserve_df):,} reserve updates to {RESERVE_FILE}")


if __name__ == "__main__":
    main()

Resuming from block 25,200,002
[100.0%] blocks 25,200,002-25,218,797 |  9036 events (total 9,036) |  9437 reserve updates (total 9,437) | ETA 0.0 h
Saved 9,036 events to s3://bastienlevyguinot/diffusion/aave_v3_events.parquet
event_name
Withdraw           3311
Supply             3294
Borrow             1350
Repay              1056
LiquidationCall      25
Saved 9,437 reserve updates to s3://bastienlevyguinot/aave_v3/aave_v3_reserve_data.parquet


fetch_transfers_etherscan.py

In [ ]:
S3_ENDPOINT = "https://minio.lab.sspcloud.fr"

def load_credentials():
    """Read the Etherscan key and the Onyxia S3 keys from Colab secrets"""
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID").strip()
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY").strip()
    os.environ["AWS_SESSION_TOKEN"]     = userdata.get("AWS_SESSION_TOKEN").strip()
    os.environ["AWS_ENDPOINT_URL"]      = S3_ENDPOINT
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
    return userdata.get("ETHERSCAN_KEY").strip()

ETHERSCAN_BASE_URL = "https://api.etherscan.io/v2/api"
ETH_CHAIN_ID = 1
# START_BLOCK  = 16_496_817  # first Aave V3 Pool event
START_BLOCK  = 25,200,000 # used only for testing

EVENTS_FILE = "s3://bastienlevyguinot/diffusion/aave_v3_events.parquet"
OUT_DIR     = "s3://bastienlevyguinot/aave_v3/transfers_parts"
OUT_FILE    = "s3://bastienlevyguinot/diffusion/transfers.parquet"
SKIPPED_FILE = "s3://bastienlevyguinot/diffusion/skipped_users_during_transfers_fetching.parquet"

BATCH_SIZE    = 1_000   # users per saved Parquet part
MAX_ROWS      = 1_000   # rows per call
CALLS_PER_SEC = 2.8     # free plan: 3 calls/s, keep a margin
WORKERS       = 4       # parallel calls (the rate limiter is shared)
MAX_CALLS_PER_USER = 20 # contracts/bots (burn address, DEX pools, MEV) have millions of transfers:
                        # an address still truncated after 20 calls (~20,000 transfers) is skipped

FIELDS = {
    "blockNumber":      "block_number",
    "timeStamp":        "timestamp",
    "hash":             "tx_hash",
    "from":             "from",
    "to":               "to",
    "contractAddress":  "token_address",
    "tokenSymbol":      "token_symbol",
    "tokenName":        "token_name",
    "tokenDecimal":     "token_decimal",
    "value":            "value",
}
SCHEMA = pa.schema([
    ("user",           pa.string()),
    ("block_number",   pa.int64()),
    ("timestamp",      pa.int64()),
    ("tx_hash",        pa.string()),
    ("from",           pa.string()),
    ("to",             pa.string()),
    ("token_address",  pa.string()),
    ("token_symbol",   pa.string()),
    ("token_name",     pa.string()),
    ("token_decimal",  pa.int64()),
    ("value",          pa.string()), # value is kept exact as a string (divide by 10**token_decimal for token units)
])
INT_COLUMNS = [f.name for f in SCHEMA if f.type == pa.int64()]

# Addresses with no transfer saved, so that they are not mistaken for users with zero transfers.
# reason: "too_many_transfers" (still truncated after MAX_CALLS_PER_USER calls)
#      or "big_block" (MAX_ROWS+ transfers in a single block)
SKIPPED_SCHEMA = pa.schema([
    ("user",           pa.string()),
    ("reason",         pa.string()),
    ("transfers_seen", pa.int64()),  # complete-block transfers fetched before giving up
    ("block_reached",  pa.int64()),  # block where the fetching stopped
])

NULL_ADDRESS = "0x0000000000000000000000000000000000000000"


class RateLimiter:

    def __init__(self, calls_per_sec):
        self.interval = 1 / calls_per_sec
        self.next_time = 0.0
        self.lock = threading.Lock()

    def wait(self):
        with self.lock:
            now = time.monotonic()
            wait = self.next_time - now
            self.next_time = max(now, self.next_time) + self.interval
        if wait > 0:
            time.sleep(wait)


def call(session, limiter, api_key, params, max_retries=10):
    """One Etherscan call with backoff on rate limits and errors"""
    params = {"chainid": ETH_CHAIN_ID, "apikey": api_key, **params}
    attempt = 0
    while True:
        limiter.wait()
        try:
            r = session.get(ETHERSCAN_BASE_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
        except (requests.RequestException, ValueError) as e:
            error = str(e)
        else:
            result = data.get("result")
            if data.get("status") == "1" or isinstance(result, list):
                return result  # "No transactions found" comes with status 0 and an empty list
            if "jsonrpc" in data and "error" not in data and isinstance(result, str) and result.startswith("0x"):
                return result  # proxy module answers in JSON-RPC format, without a status field
            error = f"{data.get('message')}: {result}"
            if "daily" in error.lower():
                print(f"Daily quota reached ({error}), waiting 1 h", flush=True)
                time.sleep(3600)
                continue
            if "invalid api key" in error.lower():
                raise SystemExit(f"Etherscan refused the key: {error}")
        attempt += 1
        if attempt == max_retries:
            raise RuntimeError(f"Etherscan call failed for {params.get('address')}: {error}")
        time.sleep(min(2 ** attempt, 60) * random.uniform(0.5, 1.5))


def fetch_user(session, limiter, api_key, user, end_block):
    """All ERC-20 transfers of one address between START_BLOCK and end_block.
    Returns (rows, None) when the address is complete, or ([], skip) when it is skipped: an address is either complete or absent, never partial.
    """
    rows = []
    start = START_BLOCK
    for _ in range(MAX_CALLS_PER_USER):
        result = call(session, limiter, api_key, {
            "module": "account", "action": "tokentx", "address": user,
            "startblock": start, "endblock": end_block,
            "page": 1, "offset": MAX_ROWS, "sort": "asc",
        })
        if len(result) < MAX_ROWS:
            rows += result
            return [{"user": user, **{col: t.get(field) for field, col in FIELDS.items()}} for t in rows], None
        # Truncated: keep the complete blocks, start again from the last one
        last = int(result[-1]["blockNumber"])
        complete = [t for t in result if int(t["blockNumber"]) < last]
        if not complete:  # MAX_ROWS+ transfers in a single block: a contract, not a wallet
            return [], {"user": user, "reason": "big_block", "transfers_seen": len(rows), "block_reached": last}
        rows += complete
        start = last
    return [], {"user": user, "reason": "too_many_transfers", "transfers_seen": len(rows), "block_reached": start}


def to_table(rows):
    df = pd.DataFrame(rows, columns=SCHEMA.names)
    for col in INT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False)


def saved_parts(fs, prefix):
    """Indexes of the parts <prefix>_XXXXX.parquet already saved in OUT_DIR."""
    fs.invalidate_cache(OUT_DIR)
    pattern = re.compile(rf"{prefix}_(\d+)\.parquet$")
    return {int(m.group(1)) for p in fs.glob(f"{OUT_DIR}/{prefix}_*.parquet") if (m := pattern.search(p))}


def merge_parts(fs, prefix, n_batches, schema, out_file):
    """Concatenate the parts into out_file one at a time, to keep memory low."""
    n_rows = 0
    with fs.open(out_file, "wb") as f, pq.ParquetWriter(f, schema) as writer:
        for i in range(n_batches):
            with fs.open(f"{OUT_DIR}/{prefix}_{i:05d}.parquet", "rb") as part:
                table = pq.read_table(part, schema=schema)
            writer.write_table(table)
            n_rows += table.num_rows
    return n_rows


def main():
    api_key = load_credentials()
    if not api_key:
        raise SystemExit("ETHERSCAN_KEY is empty: check the Colab secret")

    session = requests.Session()
    limiter = RateLimiter(CALLS_PER_SEC)
    end_block = int(call(session, limiter, api_key, {"module": "proxy", "action": "eth_blockNumber"}), 16)

    users = pd.read_parquet(EVENTS_FILE, columns=["user"])["user"].dropna().unique()
    users = sorted(u for u in users if u.lower() != NULL_ADDRESS)
    print(f"{len(users):,} users, blocks {START_BLOCK:,}-{end_block:,}")

    def fetch(user):
        return fetch_user(session, limiter, api_key, user, end_block)

    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})  # keys from the AWS_* env vars set above
    batches = [users[i:i + BATCH_SIZE] for i in range(0, len(users), BATCH_SIZE)]
    done = saved_parts(fs, "batch") & saved_parts(fs, "skipped")
    todo = [i for i in range(len(batches)) if i not in done]
    if done:
        print(f"Resuming: {len(done)} of {len(batches)} batches already saved")

    n_transfers = n_skipped = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for k, i in enumerate(todo, 1):
            results = list(executor.map(fetch, batches[i]))
            table = to_table([r for user_rows, _ in results for r in user_rows])
            skipped = pa.Table.from_pylist([s for _, s in results if s], schema=SKIPPED_SCHEMA)
            with fs.open(f"{OUT_DIR}/skipped_{i:05d}.parquet", "wb") as f:
                pq.write_table(skipped, f)
            with fs.open(f"{OUT_DIR}/batch_{i:05d}.parquet", "wb") as f:
                pq.write_table(table, f)

            n_transfers += table.num_rows
            n_skipped += skipped.num_rows
            elapsed = time.time() - t0
            eta_h = elapsed / k * (len(todo) - k) / 3600
            print(f"[{(len(done) + k) / len(batches):6.1%}] batch {i:05d} | "
                  f"{table.num_rows:7,d} transfers (total {n_transfers:,}) | "
                  f"{skipped.num_rows:3d} skipped (total {n_skipped:,}) | ETA {eta_h:.1f} h", flush=True)

    n_rows = merge_parts(fs, "batch", len(batches), SCHEMA, OUT_FILE)
    print(f"Saved {n_rows:,} transfers to {OUT_FILE}")
    n_rows = merge_parts(fs, "skipped", len(batches), SKIPPED_SCHEMA, SKIPPED_FILE)
    print(f"Saved {n_rows:,} skipped users to {SKIPPED_FILE}")


if __name__ == "__main__":
    main()

fetch_eth_etherscan.py

In [ ]:
S3_ENDPOINT = "https://minio.lab.sspcloud.fr"

def load_credentials():
    """Read the Etherscan key and the Onyxia S3 keys from Colab secrets"""
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID").strip()
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY").strip()
    os.environ["AWS_SESSION_TOKEN"]     = userdata.get("AWS_SESSION_TOKEN").strip()
    os.environ["AWS_ENDPOINT_URL"]      = S3_ENDPOINT
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
    return userdata.get("ETHERSCAN_KEY").strip()

ETHERSCAN_BASE_URL = "https://api.etherscan.io/v2/api"
ETH_CHAIN_ID = 1
# START_BLOCK  = 16_496_817  # first Aave V3 Pool event
START_BLOCK  = 25,200,000 # used only for testing

WALLETS_EVENTS_FILE = "s3://bastienlevyguinot/diffusion/aave_v3_events.parquet"
TRANSFERS_FILE      = "s3://bastienlevyguinot/diffusion/transfers_wallets.parquet"
OUT_DIR      = "s3://bastienlevyguinot/aave_v3/eth_parts"
OUT_FILE     = "s3://bastienlevyguinot/diffusion/eth_transfers_wallets.parquet"
SKIPPED_FILE = "s3://bastienlevyguinot/diffusion/skipped_users_during_eth_fetching.parquet"

BATCH_SIZE    = 1_000   # users per saved Parquet part
MAX_ROWS      = 1_000   # rows per call
CALLS_PER_SEC = 2.8     # free plan: 3 calls/s, keep a margin
WORKERS       = 4       # parallel calls (the rate limiter is shared)
MAX_CALLS_PER_USER = 20 # per (user, kind): an address still truncated after 20 calls (~20,000 rows) is skipped

ACTIONS = {"normal": "txlist", "internal": "txlistinternal"}

FIELDS = {
    "blockNumber":     "block_number",
    "timeStamp":       "timestamp",
    "hash":            "tx_hash",
    "from":            "from",
    "to":              "to",
    "value":           "value",             # in wei, kept as a string
    "isError":         "is_error",
    "methodId":        "method_id",         # normal only
    "gasUsed":         "gas_used",
    "gasPrice":        "gas_price",         # normal only
    "traceId":         "trace_id",          # internal only: position of the call in the transaction
    "type":            "call_type",         # internal only: call, create, ...
    "contractAddress": "contract_address",  # address created by the transaction, if any
}
SCHEMA = pa.schema([
    ("user",             pa.string()),
    ("kind",             pa.string()),
    ("block_number",     pa.int64()),
    ("timestamp",        pa.int64()),
    ("tx_hash",          pa.string()),
    ("from",             pa.string()),
    ("to",               pa.string()),
    ("value",            pa.string()),
    ("is_error",         pa.int64()),
    ("method_id",        pa.string()),
    ("gas_used",         pa.string()),
    ("gas_price",        pa.string()),
    ("trace_id",         pa.string()),
    ("call_type",        pa.string()),
    ("contract_address", pa.string()),
])
INT_COLUMNS = [f.name for f in SCHEMA if f.type == pa.int64()]
KIND_ONLY_COLUMNS = ["method_id", "gas_price", "trace_id", "call_type", "contract_address"]

# Addresses with no row saved, so that they are not mistaken for users with no ETH flow.
# reason: "too_many_rows" (still truncated after MAX_CALLS_PER_USER calls)
#      or "big_block" (MAX_ROWS+ rows in a single block)
SKIPPED_SCHEMA = pa.schema([
    ("user",          pa.string()),
    ("kind",          pa.string()),  # the kind that made the user skipped
    ("reason",        pa.string()),
    ("rows_seen",     pa.int64()),   # complete-block rows fetched before giving up
    ("block_reached", pa.int64()),   # block where the fetching stopped
])

NULL_ADDRESS = "0x0000000000000000000000000000000000000000"


class RateLimiter:

    def __init__(self, calls_per_sec):
        self.interval = 1 / calls_per_sec
        self.next_time = 0.0
        self.lock = threading.Lock()

    def wait(self):
        with self.lock:
            now = time.monotonic()
            wait = self.next_time - now
            self.next_time = max(now, self.next_time) + self.interval
        if wait > 0:
            time.sleep(wait)


def call(session, limiter, api_key, params, max_retries=10):
    """One Etherscan call with backoff on rate limits and errors"""
    params = {"chainid": ETH_CHAIN_ID, "apikey": api_key, **params}
    attempt = 0
    while True:
        limiter.wait()
        try:
            r = session.get(ETHERSCAN_BASE_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
        except (requests.RequestException, ValueError) as e:
            error = str(e)
        else:
            result = data.get("result")
            if data.get("status") == "1" or isinstance(result, list):
                return result  # "No transactions found" comes with status 0 and an empty list
            error = f"{data.get('message')}: {result}"
            if "daily" in error.lower():
                print(f"Daily quota reached ({error}), waiting 1 h", flush=True)
                time.sleep(3600)
                continue
            if "invalid api key" in error.lower():
                raise SystemExit(f"Etherscan refused the key: {error}")
        attempt += 1
        if attempt == max_retries:
            raise RuntimeError(f"Etherscan call failed for {params.get('address')}: {error}")
        time.sleep(min(2 ** attempt, 60) * random.uniform(0.5, 1.5))


def fetch_kind(session, limiter, api_key, kind, user, end_block):
    """Rows of one kind for one address between START_BLOCK and end_block.
    Returns (rows, None) when complete, or ([], skip) when the address is skipped.
    """
    rows = []
    start = START_BLOCK
    for _ in range(MAX_CALLS_PER_USER):
        result = call(session, limiter, api_key, {
            "module": "account", "action": ACTIONS[kind], "address": user,
            "startblock": start, "endblock": end_block,
            "page": 1, "offset": MAX_ROWS, "sort": "asc",
        })
        if len(result) < MAX_ROWS:
            rows += result
            return [{"user": user, "kind": kind, **{col: t.get(field) for field, col in FIELDS.items()}}
                    for t in rows], None
        # Truncated: keep the complete blocks, start again from the last one
        last = int(result[-1]["blockNumber"])
        complete = [t for t in result if int(t["blockNumber"]) < last]
        if not complete:  # MAX_ROWS+ rows in a single block
            return [], {"user": user, "kind": kind, "reason": "big_block",
                        "rows_seen": len(rows), "block_reached": last}
        rows += complete
        start = last
    return [], {"user": user, "kind": kind, "reason": "too_many_rows",
                "rows_seen": len(rows), "block_reached": start}


def fetch_user(session, limiter, api_key, user, end_block):
    """Normal then internal rows of one address.
    Returns (rows, None) when both kinds are complete, or ([], skip) as soon as one kind is skipped.
    """
    rows = []
    for kind in ACTIONS:
        kind_rows, skip = fetch_kind(session, limiter, api_key, kind, user, end_block)
        if skip:
            return [], skip
        rows += kind_rows
    return rows, None


def to_table(rows):
    df = pd.DataFrame(rows, columns=SCHEMA.names)
    for col in INT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in KIND_ONLY_COLUMNS:
        df[col] = df[col].replace("", None)  # fields that do not apply to the row's kind
    return pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False)


def last_block(fs, path):
    """Largest block_number of a Parquet file, from its row group statistics."""
    with fs.open(path, "rb") as f:
        meta = pq.ParquetFile(f).metadata
    col = meta.schema.names.index("block_number")
    return max(meta.row_group(i).column(col).statistics.max for i in range(meta.num_row_groups))


def saved_parts(fs, prefix):
    """Indexes of the parts <prefix>_XXXXX.parquet already saved in OUT_DIR."""
    fs.invalidate_cache(OUT_DIR)
    pattern = re.compile(rf"{prefix}_(\d+)\.parquet$")
    return {int(m.group(1)) for p in fs.glob(f"{OUT_DIR}/{prefix}_*.parquet") if (m := pattern.search(p))}


def merge_parts(fs, prefix, n_batches, schema, out_file):
    """Concatenate the parts into out_file one at a time, to keep memory low."""
    n_rows = 0
    with fs.open(out_file, "wb") as f, pq.ParquetWriter(f, schema) as writer:
        for i in range(n_batches):
            with fs.open(f"{OUT_DIR}/{prefix}_{i:05d}.parquet", "rb") as part:
                table = pq.read_table(part, schema=schema)
            writer.write_table(table)
            n_rows += table.num_rows
    return n_rows


def main():
    api_key = load_credentials()
    if not api_key:
        raise SystemExit("ETHERSCAN_KEY is empty: check the Colab secret")

    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})  # keys from the AWS_* env vars set above
    end_block = last_block(fs, TRANSFERS_FILE)
    users = pd.read_parquet(WALLETS_EVENTS_FILE, columns=["user"])["user"].dropna().unique()
    users = sorted(u for u in users if u.lower() != NULL_ADDRESS)
    print(f"{len(users):,} users, blocks {START_BLOCK:,}-{end_block:,}")

    session = requests.Session()
    limiter = RateLimiter(CALLS_PER_SEC)

    def fetch(user):
        return fetch_user(session, limiter, api_key, user, end_block)

    batches = [users[i:i + BATCH_SIZE] for i in range(0, len(users), BATCH_SIZE)]
    done = saved_parts(fs, "batch") & saved_parts(fs, "skipped")
    todo = [i for i in range(len(batches)) if i not in done]
    if done:
        print(f"Resuming: {len(done)} of {len(batches)} batches already saved")

    n_flows = n_skipped = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        for k, i in enumerate(todo, 1):
            results = list(executor.map(fetch, batches[i]))
            table = to_table([r for user_rows, _ in results for r in user_rows])
            skipped = pa.Table.from_pylist([s for _, s in results if s], schema=SKIPPED_SCHEMA)
            with fs.open(f"{OUT_DIR}/skipped_{i:05d}.parquet", "wb") as f:
                pq.write_table(skipped, f)
            with fs.open(f"{OUT_DIR}/batch_{i:05d}.parquet", "wb") as f:
                pq.write_table(table, f)

            n_flows += table.num_rows
            n_skipped += skipped.num_rows
            elapsed = time.time() - t0
            eta_h = elapsed / k * (len(todo) - k) / 3600
            print(f"[{(len(done) + k) / len(batches):6.1%}] batch {i:05d} | "
                  f"{table.num_rows:7,d} rows (total {n_flows:,}) | "
                  f"{skipped.num_rows:3d} skipped (total {n_skipped:,}) | ETA {eta_h:.1f} h", flush=True)

    n_rows = merge_parts(fs, "batch", len(batches), SCHEMA, OUT_FILE)
    print(f"Saved {n_rows:,} ETH flows to {OUT_FILE}")
    n_rows = merge_parts(fs, "skipped", len(batches), SKIPPED_SCHEMA, SKIPPED_FILE)
    print(f"Saved {n_rows:,} skipped users to {SKIPPED_FILE}")


if __name__ == "__main__":
    main()

preprocess_events_and_transfers.py

In [ ]:
S3_ENDPOINT = "https://minio.lab.sspcloud.fr"

def load_credentials():
    """Read the Onyxia S3 keys from Colab secrets (eth_getCode uses a public RPC: no API key needed)"""
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID").strip()
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY").strip()
    os.environ["AWS_SESSION_TOKEN"]     = userdata.get("AWS_SESSION_TOKEN").strip()
    os.environ["AWS_ENDPOINT_URL"]      = S3_ENDPOINT
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

EVENTS_FILE        = "s3://bastienlevyguinot/diffusion/aave_v3_events.parquet"
FETCH_SKIPPED_FILE = "s3://bastienlevyguinot/diffusion/skipped_users_during_transfers_fetching.parquet"
IN_DIR             = "s3://bastienlevyguinot/aave_v3/transfers_parts"
OUT_DIR            = "s3://bastienlevyguinot/aave_v3/transfers_parts_preprocessed"
CODE_DIR           = "s3://bastienlevyguinot/aave_v3/code_types_parts"
TRANSFERS_OUT      = "s3://bastienlevyguinot/diffusion/transfers_preprocessed.parquet"
EVENTS_OUT         = "s3://bastienlevyguinot/diffusion/aave_v3_events_preprocessed.parquet"
SKIPPED_OUT        = "s3://bastienlevyguinot/diffusion/skipped_addresses.parquet"

EVENT_ADDRESS_COLUMNS    = ["reserve", "user", "caller", "recipient", "repayer", "collateral_asset", "liquidator"]
TRANSFER_ADDRESS_COLUMNS = ["user", "from", "to", "token_address"]

BURN_ADDRESSES = {
    "0x0000000000000000000000000000000000000000",  # null address (mint / burn)
    "0x000000000000000000000000000000000000dead",
    "0xdead000000000000000042069420694206942069",
    *(f"0x{i:040x}" for i in range(1, 11)),        # precompiles 0x01-0x0a
}

RPC_URL   = "https://ethereum.publicnode.com"
RPC_BATCH = 50      # eth_getCode calls per HTTP request
PART_SIZE = 5_000   # addresses per cached Parquet part
KEPT_CODE_TYPES = {"eoa", "eoa_7702", "safe"}
SAFE_SELECTOR = "a619486e"  # keccak256("masterCopy()")

SKIPPED_SCHEMA = pa.schema([
    ("address",     pa.string()),
    ("reason",      pa.string()),
    ("n_transfers", pa.int64()),
    ("n_events",    pa.int64()),
])

BATCH_RE = re.compile(r"batch_(\d+)\.parquet$")
PART_RE  = re.compile(r"part_(\d+)\.parquet$")


# ---------------------------------------------------------------- inputs

def load_events(fs):
    """Returns every event, with lowercased address columns."""
    with fs.open(EVENTS_FILE, "rb") as f:
        events = pd.read_parquet(f)
    for col in EVENT_ADDRESS_COLUMNS:
        if col in events:
            events[col] = events[col].str.lower()
    return events


def load_fetch_skipped(fs):
    """Returns a set of lowercased addresses skipped by fetch_transfers_etherscan.py."""
    if not fs.exists(FETCH_SKIPPED_FILE):
        raise SystemExit(f"{FETCH_SKIPPED_FILE} not found: run fetch_transfers_etherscan.py to the end first")
    with fs.open(FETCH_SKIPPED_FILE, "rb") as f:
        return set(pd.read_parquet(f, columns=["user"])["user"].str.lower())


def scan_transfers(fs, paths):
    """Lowercased token addresses, and user -> (n_transfers, last_transfer_block), one batch at a time."""
    tokens, stats = set(), []
    for path in paths:
        with fs.open(path, "rb") as f:
            df = pq.read_table(f, columns=["user", "token_address", "block_number"]).to_pandas()
        tokens.update(df["token_address"].dropna().str.lower())
        df["user"] = df["user"].str.lower()
        stats.append(df.groupby("user")["block_number"].agg(n_transfers="size", last_transfer_block="max"))
    if not stats:
        return tokens, pd.DataFrame(columns=["n_transfers", "last_transfer_block"])
    stats = pd.concat(stats)
    return tokens, stats.groupby(level=0).agg({"n_transfers": "sum", "last_transfer_block": "max"})


# ---------------------------------------------------------------- eth_getCode

def code_type(code):
    code = code.lower()
    if code in ("0x", ""):
        return "eoa"
    if code.startswith("0xef0100") and len(code) == 2 + 2 * 23:
        return "eoa_7702"
    if SAFE_SELECTOR in code and len(code) < 2 + 2 * 1_000:  # proxies are small
        return "safe"
    return "contract"


def get_codes(session, addresses, max_retries=10):
    """address -> code, retrying the calls of the batch that failed"""
    codes, todo, error = {}, list(addresses), None
    for attempt in range(max_retries):
        payload = [{"jsonrpc": "2.0", "id": i, "method": "eth_getCode", "params": [a, "latest"]}
                   for i, a in enumerate(todo)]
        try:
            answers = session.post(RPC_URL, json=payload, timeout=60).json()
        except (requests.RequestException, ValueError) as e:
            answers, error = [], str(e)
        if isinstance(answers, dict):  # the whole batch was refused
            answers, error = [], str(answers.get("error"))
        for answer in answers:
            if "result" in answer:
                codes[todo[answer["id"]]] = answer["result"]
            else:
                error = str(answer.get("error"))
        todo = [a for a in todo if a not in codes]
        if not todo:
            return codes
        time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f"eth_getCode failed for {len(todo)} addresses: {error}")


def load_code_types(fs):
    """Cached code types and the index of the next part to write."""
    fs.invalidate_cache(CODE_DIR)
    paths = [p for p in fs.glob(f"{CODE_DIR}/part_*.parquet") if PART_RE.search(p)]
    parts = []
    for path in paths:
        with fs.open(path, "rb") as f:
            parts.append(pd.read_parquet(f, columns=["address", "code_type"]))
    cached = pd.concat(parts) if parts else pd.DataFrame(columns=["address", "code_type"])
    cached["address"] = cached["address"].str.lower()
    next_part = max((int(PART_RE.search(p).group(1)) for p in paths), default=-1) + 1
    return cached.drop_duplicates("address"), next_part


def fetch_code_types(fs, users):
    """address -> code_type for every user, using the cache in CODE_DIR."""
    cached, next_part = load_code_types(fs)
    todo = sorted(set(users) - set(cached["address"]))
    print(f"eth_getCode: {len(users):,} users, {len(users) - len(todo):,} cached, {len(todo):,} to fetch")

    session = requests.Session()
    parts = [cached]
    t0 = time.time()
    for k, start in enumerate(range(0, len(todo), PART_SIZE)):
        chunk = todo[start:start + PART_SIZE]
        codes = {}
        for i in range(0, len(chunk), RPC_BATCH):
            codes.update(get_codes(session, chunk[i:i + RPC_BATCH]))
        part = pd.DataFrame({
            "address":   chunk,
            "code_type": [code_type(codes[a]) for a in chunk],
            "code_size": [(len(codes[a]) - 2) // 2 for a in chunk],
        })
        with fs.open(f"{CODE_DIR}/part_{next_part + k:05d}.parquet", "wb") as f:
            part.to_parquet(f, index=False)
        parts.append(part[["address", "code_type"]])
        done = start + len(chunk)
        eta_min = (time.time() - t0) / done * (len(todo) - done) / 60
        print(f"[{done / len(todo):6.1%}] {done:,} code types fetched | ETA {eta_min:.0f} min", flush=True)
    types = pd.concat(parts, ignore_index=True).set_index("address")["code_type"]
    return types.reindex(sorted(users))


# ---------------------------------------------------------------- rules

def add_reasons(reason, rules):
    """Append each rule's name to reason where its mask is True, joined with "+" """
    for name, mask in rules:
        reason = reason.mask(mask, (reason + "+" + name).str.lstrip("+"))
    return reason


def classify_users(fs, events, fetch_skipped, tokens, stats):
    """One row per user of the events: n_events, n_transfers and reason ("" if kept)"""
    users = (events.dropna(subset=["user"]).groupby("user")["block_number"].agg(n_events="size", last_event_block="max"))
    users = users.join(stats, how="left")
    users["n_transfers"] = users["n_transfers"].fillna(0).astype("int64")
    address = users.index.to_series()

    # 1. Identity
    reserves = set(pd.concat([events["reserve"], events["collateral_asset"]]).dropna())
    reason = add_reasons(pd.Series("", index=users.index), [
        ("skipped_fetch", address.isin(fetch_skipped)),
        ("burn",          address.isin(BURN_ADDRESSES)),
        ("reserve",       address.isin(reserves)),
        ("token",         address.isin(tokens)),
    ])

    # 2. Consistency
    left = reason == ""
    no_transfers = left & (users["n_transfers"] == 0)
    after_last = (users["last_event_block"] > users["last_transfer_block"]).fillna(False).astype(bool)
    reason[no_transfers] = "no_transfers"
    reason[left & ~no_transfers & after_last] = "event_after_last_transfer"

    # 3. Code
    left = reason == ""
    types = fetch_code_types(fs, list(users.index[left]))
    print(types.value_counts().to_string())
    contracts = types.index[~types.isin(KEPT_CODE_TYPES)]
    reason.loc[contracts] = "contract"

    users["reason"] = reason
    return users


# ---------------------------------------------------------------- outputs

def lower_columns(table, cols):
    for col in cols:
        i = table.schema.get_field_index(col)
        if i >= 0:
            table = table.set_column(i, col, pc.utf8_lower(table.column(col)))
    return table


def write_transfers(fs, paths, kept):
    """Batches without the dropped users to OUT_DIR, merged into TRANSFERS_OUT; returns the rows written."""
    kept = pa.array(sorted(kept), pa.string())
    names, n_rows, writer = set(), 0, None
    with fs.open(TRANSFERS_OUT, "wb") as out:
        for path in paths:
            with fs.open(path, "rb") as f:
                table = lower_columns(pq.read_table(f), TRANSFER_ADDRESS_COLUMNS)
            table = table.filter(pc.is_in(table.column("user"), value_set=kept))
            name = path.rsplit("/", 1)[-1]
            with fs.open(f"{OUT_DIR}/{name}", "wb") as f:
                pq.write_table(table, f)
            if writer is None:
                writer = pq.ParquetWriter(out, table.schema)
            writer.write_table(table)
            names.add(name)
            n_rows += table.num_rows
        writer.close()

    # Batches left by an earlier run on more raw batches
    fs.invalidate_cache(OUT_DIR)
    for p in fs.glob(f"{OUT_DIR}/batch_*.parquet"):
        if p.rsplit("/", 1)[-1] not in names:
            fs.rm(p)
    return n_rows


def main():
    load_credentials()
    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})  # keys from the AWS_* env vars set above
    fs.invalidate_cache(IN_DIR)
    paths = sorted(p for p in fs.glob(f"{IN_DIR}/batch_*.parquet") if BATCH_RE.search(p))
    if not paths:
        raise SystemExit(f"No batch in {IN_DIR}: run fetch_transfers_etherscan.py first")

    events = load_events(fs)
    fetch_skipped = load_fetch_skipped(fs)
    tokens, stats = scan_transfers(fs, paths)
    print(f"{len(events):,} events, {len(paths)} transfer batches ({stats['n_transfers'].sum():,} transfers), "
          f"{len(tokens):,} tokens, {len(fetch_skipped):,} users skipped by the fetch")

    users = classify_users(fs, events, fetch_skipped, tokens, stats)
    kept = set(users.index[users["reason"] == ""])
    dropped = users[users["reason"] != ""]

    n_rows = write_transfers(fs, paths, kept)
    print(f"{len(paths)} batches written to {OUT_DIR}, {n_rows:,} transfers saved to {TRANSFERS_OUT}")

    is_kept = events["user"].isin(kept)
    with fs.open(EVENTS_OUT, "wb") as f:
        events[is_kept].to_parquet(f, index=False)
    print(f"{is_kept.sum():,} of {len(events):,} events saved to {EVENTS_OUT} "
          f"({events['user'].isna().sum():,} without user dropped)")
    print(pd.DataFrame({"kept": events.loc[is_kept, "event_name"].value_counts(),
                        "dropped": events.loc[~is_kept, "event_name"].value_counts()}).fillna(0).astype(int).to_string())

    skipped = (dropped.rename_axis("address").reset_index()[SKIPPED_SCHEMA.names]
               .sort_values(["n_transfers", "n_events"], ascending=False))
    with fs.open(SKIPPED_OUT, "wb") as f:
        pq.write_table(pa.Table.from_pandas(skipped, schema=SKIPPED_SCHEMA, preserve_index=False), f)
    print(f"{len(kept):,} users kept, {len(skipped):,} dropped, saved to {SKIPPED_OUT}")
    print(skipped.groupby("reason").agg(users=("address", "size"), n_transfers=("n_transfers", "sum"),
                                        n_events=("n_events", "sum")).to_string())


if __name__ == "__main__":
    main()

match_events_transfers.py (ce code est voué à être modifié dans les prochaines versions)

In [ ]:
S3_ENDPOINT = "https://minio.lab.sspcloud.fr"

def load_credentials():
    """Read the Onyxia S3 keys from Colab secrets (getReserveData uses a public RPC: no API key needed)"""
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID").strip()
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY").strip()
    os.environ["AWS_SESSION_TOKEN"]     = userdata.get("AWS_SESSION_TOKEN").strip()
    os.environ["AWS_ENDPOINT_URL"]      = S3_ENDPOINT
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

EVENTS_FILE = "s3://bastienlevyguinot/diffusion/aave_v3_events_wallets.parquet"
WALLETS_DIR = "s3://bastienlevyguinot/aave_v3/transfers_parts_wallets"
OUT_FILE    = "s3://bastienlevyguinot/diffusion/transfers_wallets.parquet"
CHECKS_FILE = "s3://bastienlevyguinot/aave_v3/events_transfers_checks.parquet"

RPC_URL      = "https://ethereum.publicnode.com"
POOL_ADDRESS = "0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2"  # Aave V3 Pool (proxy)
GET_RESERVE_DATA = "0x35ea6a75"  # getReserveData(address)

NULL_ADDRESS = "0x0000000000000000000000000000000000000000"
Z = NULL_ADDRESS

BATCH_RE = re.compile(r"batch_(\d+)\.parquet$")


def reserve_tokens(reserves):
    """Lowercased reserve -> (aToken, variable debt token), words 8 and 10 of getReserveData"""
    payload = [{"jsonrpc": "2.0", "id": i, "method": "eth_call",
                "params": [{"to": POOL_ADDRESS, "data": GET_RESERVE_DATA + r[2:].rjust(64, "0")}, "latest"]}
               for i, r in enumerate(reserves)]
    answers = requests.post(RPC_URL, json=payload, timeout=60).json()
    tokens = {}
    for answer in answers:
        words = answer["result"][2:]
        word = lambda k: "0x" + words[64 * k + 24:64 * (k + 1)]
        tokens[reserves[answer["id"]]] = (word(8), word(10))
    return tokens


def load_events():
    """Events with lowercased addresses and the aToken / debt token of their reserves."""
    ev = pd.read_parquet(EVENTS_FILE)
    ev["event_id"] = range(len(ev))
    for col in ["tx_hash", "reserve", "caller", "recipient", "repayer", "liquidator", "collateral_asset"]:
        ev[col] = ev[col].str.lower()
    ev["u"] = ev["user"].str.lower()
    reserves = sorted(set(ev["reserve"].dropna()) | set(ev["collateral_asset"].dropna()))
    tokens = reserve_tokens(reserves)
    ev["a_token"] = ev["reserve"].map(lambda r: tokens[r][0])
    ev["d_token"] = ev["reserve"].map(lambda r: tokens[r][1])
    ev["a_collateral"] = ev["collateral_asset"].map(lambda r: tokens[r][0], na_action="ignore")
    return ev


def flow(m, token, src, dst):
    """Transfers of token from src to dst (columns of m or a constant address)."""
    col = lambda x: m[x] if x in m else x
    return (m["token"] == col(token)) & (m["src"] == col(src)) & (m["dst"] == col(dst))


def mint_or_burn(m, token):
    return flow(m, token, "u", Z) | flow(m, token, Z, "u")


# rule -> (event, is the transfer expected for this event, does a transfer row satisfy it)
RULES = {
    "s_mint":       ("Supply",   lambda e: True,
                     lambda m: flow(m, "a_token", Z, "u")),
    "s_underlying": ("Supply",   lambda e: e["caller"] == e["u"],
                     lambda m: flow(m, "reserve", "caller", "a_token") & (m["value"] == m["amount_raw"])),
    "w_burn":       ("Withdraw", lambda e: True,
                     lambda m: mint_or_burn(m, "a_token")),
    "w_underlying": ("Withdraw", lambda e: e["recipient"] == e["u"],
                     lambda m: flow(m, "reserve", "a_token", "recipient") & (m["value"] == m["amount_raw"])),
    "b_mint":       ("Borrow",   lambda e: e["interest_rate_mode"] == 2,
                     lambda m: flow(m, "d_token", Z, "u")),
    "b_underlying": ("Borrow",   lambda e: e["caller"] == e["u"],
                     lambda m: (flow(m, "reserve", "a_token", "caller") | flow(m, "reserve", Z, "caller"))
                     & (m["value"] == m["amount_raw"])),
    "r_burn":       ("Repay",    lambda e: True,
                     lambda m: mint_or_burn(m, "d_token")),
    "r_underlying": ("Repay",    lambda e: (e["repayer"] == e["u"]) & ~e["use_atokens"].fillna(False),
                     lambda m: flow(m, "reserve", "repayer", "a_token") & (m["value"] == m["amount_raw"])),
    "r_atoken":     ("Repay",    lambda e: e["use_atokens"].fillna(False),
                     lambda m: mint_or_burn(m, "a_token")),
    "l_debt_burn":  ("LiquidationCall", lambda e: True,
                     lambda m: mint_or_burn(m, "d_token")),
    "l_collateral": ("LiquidationCall", lambda e: True,
                     lambda m: ((m["token"] == m["a_collateral"]) & (m["src"] == m["u"]))
                     | flow(m, "a_collateral", Z, "u")),
}


def check_events(ev, tr):
    """One row per event: each rule's result (NA if not expected) and whether all were found."""
    t = pd.DataFrame({"tx_hash": tr["tx_hash"].str.lower(), "user": tr["user"],
                      "token": tr["token_address"].str.lower(), "src": tr["from"].str.lower(),
                      "dst": tr["to"].str.lower(), "value": tr["value"]})
    m = ev.merge(t, on=["tx_hash", "user"])
    out = ev[["event_id"]].copy()
    for rule, (name, expected, match) in RULES.items():
        found = pd.Series(False, index=ev.index)
        found_ids = set(m.loc[(m["event_name"] == name) & match(m).fillna(False), "event_id"])
        found[ev["event_id"].isin(found_ids)] = True
        is_expected = pd.Series(expected(ev), index=ev.index).fillna(False).astype(bool)
        is_expected &= ev["event_name"] == name
        out[rule] = found.astype("boolean").where(is_expected)
    out["complete"] = out[list(RULES)].fillna(True).all(axis=1)
    return out


def main():
    load_credentials()
    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})  # keys from the AWS_* env vars set above
    events = load_events()
    print(f"{len(events):,} events")

    fs.invalidate_cache(WALLETS_DIR)
    paths = sorted(p for p in fs.glob(f"{WALLETS_DIR}/batch_*.parquet") if BATCH_RE.search(p))
    checks, last_blocks = [], []
    n_rows = n_tracked = 0
    tmp_file = f"{OUT_FILE}.tmp"  # OUT_FILE is replaced only once complete
    with fs.open(tmp_file, "wb") as f:
        writer = None
        for path in paths:
            with fs.open(path, "rb") as part:
                table = pq.read_table(part)
            tr = table.to_pandas()
            ev = events[events["user"].isin(set(tr["user"]))]

            keys = pd.MultiIndex.from_frame(ev[["tx_hash", "user"]])
            tracked = pd.MultiIndex.from_arrays([tr["tx_hash"].str.lower(), tr["user"]]).isin(keys)
            table = table.append_column("tracked", pa.array(tracked, pa.bool_()))
            writer = writer or pq.ParquetWriter(f, table.schema)
            writer.write_table(table)
            n_rows += len(tr)
            n_tracked += int(tracked.sum())

            checks.append(check_events(ev, tr))
            last_blocks.append(tr.groupby("user")["block_number"].max())
        writer.close()
    fs.mv(tmp_file, OUT_FILE)
    print(f"{n_tracked:,} of {n_rows:,} transfers tracked ({n_tracked / n_rows:.1%}), saved to {OUT_FILE}")

    checks = events[["event_id", "event_name", "block_number", "tx_hash", "log_index", "user"]].merge(
        pd.concat(checks), on="event_id", how="left")
    last_block = checks["user"].map(pd.concat(last_blocks))
    checks["after_last_transfer"] = last_block.isna() | (checks["block_number"] > last_block)
    checks = checks.drop(columns="event_id")
    with fs.open(CHECKS_FILE, "wb") as f:
        checks.to_parquet(f, index=False)

    print("\nExpected transfers found, by rule")
    rows = []
    for rule, (name, _, _) in RULES.items():
        col = checks[rule].dropna()
        rows.append({"event": name, "rule": rule, "expected": len(col), "found": int(col.sum()),
                     "share": col.mean() if len(col) else float("nan")})
    print(pd.DataFrame(rows).to_string(index=False))

    print("\nComplete events, by type (before / after the last transfer fetched for the user)")
    summary = checks.groupby(["event_name", "after_last_transfer"])["complete"].agg(["size", "sum", "mean"])
    print(summary.rename(columns={"size": "events", "sum": "complete", "mean": "share"}).to_string())
    print(f"\nChecks saved to {CHECKS_FILE}")


if __name__ == "__main__":
    main()

trace_wallets.py (ce code est voué à être modifié dans les prochaines versions)

In [ ]:
S3_ENDPOINT = "https://minio.lab.sspcloud.fr"

def load_credentials():
    """Read the Onyxia S3 keys from Colab secrets (decimals() and symbol() use a public RPC: no API key needed)"""
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID").strip()
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY").strip()
    os.environ["AWS_SESSION_TOKEN"]     = userdata.get("AWS_SESSION_TOKEN").strip()
    os.environ["AWS_ENDPOINT_URL"]      = S3_ENDPOINT
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

INPUT_FILE    = "s3://bastienlevyguinot/diffusion/transfers_wallets.parquet"
EVENTS_FILE   = "s3://bastienlevyguinot/diffusion/aave_v3_events_wallets.parquet"
ETH_FILE      = "s3://bastienlevyguinot/diffusion/eth_transfers_wallets.parquet"
ETH_STATUS_DIR = "s3://bastienlevyguinot/aave_v3/eth_status_parts"
OUT_FILE      = "s3://bastienlevyguinot/diffusion/transfers_wallets_traced.parquet"
LEDGERS_FILE  = "s3://bastienlevyguinot/aave_v3/filtering_ledgers.parquet"
ORIGINAL_FILE = "s3://bastienlevyguinot/diffusion/original_portfolios.parquet"

RPC_URL = "https://ethereum.publicnode.com"

CHUNK_ROWS   = 1_000_000
MIN_FRACTION = 1e-6  # smaller traced fractions are float residue: treated as 0
DEBT_PREFIXES = ("variableDebt", "stableDebt")  # debt tokens of Aave and its forks: never own funds
WETH = "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"  # native ETH rows use this token_address


def trace_user(user, tx_hash, src, dst, token, amount, tracked, supplies=None, debt_tokens=()):
    """traced_fraction of each transfer of one user, the final ledger and the contributions to the original portfolio."""
    supplies = supplies or {}
    fractions = [0.0] * len(tx_hash)
    ledger = {}
    contributions = []

    by_tx = defaultdict(list)  # insertion order = order of first appearance
    for i, h in enumerate(tx_hash):
        by_tx[h].append(i)

    for h, rows in by_tx.items():
        out, inc = defaultdict(float), defaultdict(float)
        for i in rows:
            if src[i] == user:
                out[token[i]] += amount[i]
            if dst[i] == user:
                inc[token[i]] += amount[i]

        if tracked[rows[0]]:  # Aave transaction
            f = 1.0
            for t, a in supplies.get(h, {}).items():
                q = a - min(ledger.get(t, 0.0), out.get(t, 0.0))
                if q > 0:
                    contributions.append((rows[0], t, q, "supply"))
        else:  # weakest link over the traced tokens sent (zero amounts move nothing: ignored)
            ratios = [min(1.0, ledger[t] / a) for t, a in out.items()
                      if a > 0 and ledger.get(t, 0.0) > 0]
            f = min(ratios, default=0.0)
            if f < MIN_FRACTION:  # nothing traced sent (pure receives included), or float dust
                continue
            # untraced tokens sent alongside traced ones: f * amount of own funds consumed
            for t, a in out.items():
                if ledger.get(t, 0.0) <= 0 and a > 0 and t not in debt_tokens:
                    contributions.append((rows[0], t, f * a, "complement"))

        for t, a in out.items():
            ledger[t] = max(0.0, ledger.get(t, 0.0) - a)
        for t, a in inc.items():
            ledger[t] = ledger.get(t, 0.0) + f * a
        for i in rows:
            fractions[i] = f

    return fractions, ledger, contributions


def reserve_info(reserves):
    """Lowercased reserve -> (decimals, symbol)"""
    calls = [(r, sel) for r in reserves for sel in ("0x313ce567", "0x95d89b41")]
    results = {}
    for k in range(0, len(calls), 50):  # the node accepts batches of 50 calls
        payload = [{"jsonrpc": "2.0", "id": i, "method": "eth_call",
                    "params": [{"to": r, "data": sel}, "latest"]}
                   for i, (r, sel) in enumerate(calls[k:k + 50], start=k)]
        for answer in requests.post(RPC_URL, json=payload, timeout=60).json():
            results[calls[answer["id"]]] = bytes.fromhex(answer["result"][2:])
    info = {}
    for r in reserves:
        raw = results[(r, "0x95d89b41")]
        if len(raw) > 32:
            symbol = raw[64:64 + int.from_bytes(raw[32:64], "big")].decode(errors="replace")
        else:  # bytes32 (e.g. MKR)
            symbol = raw.rstrip(b"\0").decode(errors="replace")
        info[r] = (int.from_bytes(results[(r, "0x313ce567")], "big"), symbol)
    return info


def load_events(fs):
    """ user -> tx_hash -> {reserve: amount supplied in token units},
        reserve -> symbol,
        the last block of the events,
        the (tx_hash, user) of every event"""
    with fs.open(EVENTS_FILE, "rb") as f:
        ev = pd.read_parquet(f, columns=["event_name", "block_number", "user", "tx_hash", "reserve",
                                         "amount_raw"])
    last_block = int(ev["block_number"].max())
    ev = ev.assign(user=ev["user"].str.lower(), tx_hash=ev["tx_hash"].str.lower(),
                   reserve=ev["reserve"].str.lower())
    aave_txs = set(zip(ev["tx_hash"], ev["user"]))
    ev = ev[ev["event_name"] == "Supply"]
    info = reserve_info(sorted(ev["reserve"].unique()))
    supplies = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    for u, h, r, a in zip(ev["user"], ev["tx_hash"], ev["reserve"], ev["amount_raw"]):
        supplies[u][h][r] += int(a) / 10 ** info[r][0]  # several Supply events of one tx are summed
    return supplies, {r: sym for r, (_, sym) in info.items()}, last_block, aave_txs


def cut_eth_users(fs):
    """Lowercased users whose ETH history was cut by fetch_eth_etherscan.py."""
    cut = set()
    for path in fs.glob(f"{ETH_STATUS_DIR}/batch_*.parquet"):
        with fs.open(path, "rb") as f:
            status = pd.read_parquet(f)
        cut |= set(status.loc[status["truncated"], "user"].str.lower())
    return cut


def load_eth(fs, schema, last_block, aave_txs, skip):
    with fs.open(ETH_FILE, "rb") as f:
        t = pq.read_table(f, columns=["user", "kind", "block_number", "timestamp", "tx_hash",
                                      "from", "to", "value", "is_error"])
    weth = WETH.lower()
    keep = pc.and_(pc.and_(pc.not_equal(t["value"], "0"), pc.equal(t["is_error"], 0)),
                   pc.less_equal(t["block_number"], last_block))
    wrap = pc.or_(pc.and_(pc.equal(t["kind"], "normal"), pc.equal(pc.utf8_lower(t["to"]), weth)),
                  pc.and_(pc.equal(t["kind"], "internal"), pc.equal(pc.utf8_lower(t["from"]), weth)))
    t = t.filter(pc.and_(keep, pc.invert(wrap)))
    df = t.to_pandas()
    df["u"] = df["user"].str.lower()
    df = df[~df["u"].isin(skip)].sort_values("u", kind="stable")  # each user's order is kept
    df["tracked"] = [(h, u) in aave_txs for h, u in zip(df["tx_hash"], df["u"])]
    df = df.assign(token_address=WETH, token_symbol="ETH", token_name="Ether", token_decimal=18)
    bounds = df.reset_index(drop=True).groupby("u", sort=False).indices
    index = {u: (int(rows[0]), int(rows[-1]) + 1) for u, rows in bounds.items()}
    table = pa.Table.from_pandas(df[schema.names], schema=schema, preserve_index=False)
    return table, index


def add_eth(table, eth, eth_index):
    """table with the ETH rows and ERC-20 transfers rows, chronological order."""
    users = pc.unique(pc.utf8_lower(table.column("user"))).to_pylist()  # in order of appearance
    parts = []
    for u in users:
        if u in eth_index:
            start, end = eth_index[u]
            parts.append(eth.slice(start, end - start))
    if not parts:
        return table
    table = pa.concat_tables([table, *parts])
    rank = pc.index_in(pc.utf8_lower(table.column("user")), pa.array(users))
    order = pc.sort_indices(pa.table({"r": rank, "b": table.column("block_number")}),
                            sort_keys=[("r", "ascending"), ("b", "ascending")])  # stable
    return table.take(order)


def trace_chunk(table, seen, supplies, reserve_symbols):
    """traced_fraction of every row of a table holding whole users, their ledgers and their
    original portfolio contributions as rows, and the number of Supply transactions not found."""
    users = [u.lower() for u in table.column("user").to_pylist()]
    tx_hash = table.column("tx_hash").to_pylist()
    block = table.column("block_number").to_pylist()
    timestamp = table.column("timestamp").to_pylist()
    src = [s.lower() for s in table.column("from").to_pylist()]
    dst = [d.lower() for d in table.column("to").to_pylist()]
    token = [t.lower() for t in table.column("token_address").to_pylist()]
    symbol = table.column("token_symbol").to_pylist()
    decimals = table.column("token_decimal").to_pylist()
    amount = [float(v) / 10 ** (d or 0) for v, d in zip(table.column("value").to_pylist(), decimals)]
    tracked = table.column("tracked").to_pylist()

    fractions, ledger_rows, original_rows = [], [], []
    n_missing = 0
    start = 0
    while start < len(users):
        end = start
        while end < len(users) and users[end] == users[start]:
            end += 1
        user = users[start]
        if user in seen:
            raise RuntimeError(f"rows of {user} are not contiguous in {INPUT_FILE}")
        seen.add(user)
        s = slice(start, end)
        user_supplies = supplies.get(user, {})
        symbols = {**reserve_symbols, **dict(zip(token[s], symbol[s]))}
        if WETH.lower() in symbols:
            symbols[WETH.lower()] = "WETH+ETH"  # one ledger entry for both
        debt = {t for t, sym in symbols.items() if (sym or "").startswith(DEBT_PREFIXES)}
        f, ledger, contributions = trace_user(user, tx_hash[s], src[s], dst[s], token[s], amount[s],
                                              tracked[s], user_supplies, debt)
        n_missing += len(set(user_supplies) - set(tx_hash[s]))
        fractions += f
        ledger_rows += [(user, t, symbols[t], q) for t, q in ledger.items()]
        original_rows += [(user, block[start + i], timestamp[start + i], tx_hash[start + i],
                           t, symbols[t], q, source) for i, t, q, source in contributions]
        start = end
    return fractions, ledger_rows, original_rows, n_missing


def main():
    load_credentials()
    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT})  # keys from the AWS_* env vars set above
    supplies, reserve_symbols, last_block, aave_txs = load_events(fs)
    skip = cut_eth_users(fs)
    seen, ledger_rows, original_rows = set(), [], []
    n_rows = n_kept = n_forced = n_missing = n_eth_kept = 0
    tmp_file = f"{OUT_FILE}.tmp"  # OUT_FILE is replaced only once complete

    with fs.open(INPUT_FILE, "rb") as src_file, fs.open(tmp_file, "wb") as out_file:
        reader = pq.ParquetFile(src_file)
        eth, eth_index = load_eth(fs, reader.schema_arrow, last_block, aave_txs, skip)
        print(f"{eth.num_rows:,} ETH rows for {len(eth_index):,} users, "
              f"{len(skip)} users skipped (ETH history cut)", flush=True)
        writer = None
        pending = None

        def process(table):
            nonlocal writer, n_rows, n_kept, n_forced, n_missing, n_eth_kept
            table = add_eth(table, eth, eth_index)
            fractions, ledgers, originals, missing = trace_chunk(table, seen, supplies, reserve_symbols)
            ledger_rows.extend(ledgers)
            original_rows.extend(originals)
            n_missing += missing
            fractions = pa.array(fractions, pa.float64())
            table = table.append_column("traced_fraction", fractions)
            table = table.filter(pc.greater(fractions, 0.0))
            writer = writer or pq.ParquetWriter(out_file, table.schema)
            writer.write_table(table)
            n_rows += len(fractions)
            n_kept += table.num_rows
            n_forced += int(pc.sum(table.column("tracked")).as_py() or 0)
            native = pc.and_(pc.equal(table.column("token_address"), WETH),
                             pc.equal(table.column("token_symbol"), "ETH"))
            n_eth_kept += int(pc.sum(native).as_py() or 0)
            print(f"{n_rows:,} transfers read, {n_kept:,} kept", flush=True)

        for batch in reader.iter_batches(batch_size=CHUNK_ROWS):
            table = pa.Table.from_batches([batch])
            # after the last event, Aave transactions could not be recognised: dropped
            table = table.filter(pc.less_equal(table.column("block_number"), last_block))
            table = table.filter(pc.invert(pc.is_in(pc.utf8_lower(table.column("user")),
                                                     pa.array(sorted(skip), pa.string()))))
            if pending is not None:
                table = pa.concat_tables([pending, table])
            if not len(table):
                continue
            # Carry the last user over: its rows may continue in the next chunk
            users = table.column("user")
            last = users[-1].as_py()
            split = len(table)
            while split > 0 and users[split - 1].as_py() == last:
                split -= 1
            pending = table.slice(split)
            if split:
                process(table.slice(0, split))
        if pending is not None and len(pending):
            process(pending)
        writer.close()
    fs.mv(tmp_file, OUT_FILE)

    ledgers = pd.DataFrame(ledger_rows, columns=["user", "token_address", "token_symbol", "final_quantity"])
    with fs.open(LEDGERS_FILE, "wb") as f:
        ledgers.to_parquet(f, index=False)

    print(f"\nTransfers up to block {last_block:,} (last event)")
    print(f"{len(seen):,} users, {n_kept:,} of {n_rows:,} transfers kept ({n_kept / n_rows:.1%}), "
          f"{n_forced:,} of them forced, {n_eth_kept:,} of them ETH, saved to {OUT_FILE}")
    print(f"{len(ledgers):,} ledger entries ({(ledgers['final_quantity'] > 0).sum():,} positive), "
          f"saved to {LEDGERS_FILE}")

    original = pd.DataFrame(original_rows, columns=["user", "block_number", "timestamp", "tx_hash",
                                                    "token_address", "token_symbol", "quantity", "source"])
    with fs.open(ORIGINAL_FILE, "wb") as f:
        original.to_parquet(f, index=False)
    print(f"{len(original):,} original portfolio contributions "
          f"({original['source'].value_counts().to_dict()}), saved to {ORIGINAL_FILE}")
    if n_missing:
        print(f"WARNING: {n_missing:,} Supply transactions have no transfer of their user: ignored")


if __name__ == "__main__":
    main()